# Part 4: Using More Complex Non-Linear Models

#### Welcome to Part 4!

Parts 1–3 used a simple linear classifier on top of nnAudio2's Mel spectrogram. In this tutorial we swap in a deeper model — **BC-ResNet (Broadcasting-residual network)** — while keeping the same nnAudio2 front-end. Because the spectrogram layer is just an `nn.Module`, nothing else changes.

You can substitute any other model in Step 4 by replacing the `BCResNet` class.

[Step 1: Imports](#Step-1:-Imports)\
[Step 2: Configuration & device](#Step-2:-Configuration-&-device)\
[Step 3: Dataset & DataLoaders](#Step-3:-Dataset-&-DataLoaders)\
[Step 4: Model (BC-ResNet + nnAudio2 front-end)](#Step-4:-Model)\
[Step 5: Train & test](#Step-5:-Train-&-test)

## Step 1: Imports

In [9]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch import Tensor
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from pytorch_lightning import Trainer, LightningModule
from sklearn.metrics import precision_recall_fscore_support

from nnAudio2.features.mel import MelSpectrogram

## Step 2: Configuration & device

Device is detected automatically — CUDA → MPS (Apple Silicon) → CPU.

In [10]:
if torch.cuda.is_available():
    device, accelerator = 'cuda', 'gpu'
elif torch.backends.mps.is_available():
    device, accelerator = 'mps', 'mps'
else:
    device, accelerator = 'cpu', 'cpu'

print(f"Using device: {device}")

batch_size              = 100
max_epochs              = 200
check_val_every_n_epoch = 2
num_sanity_val_steps    = 5

data_root       = './'
download_option = True   # set False once downloaded

n_mels     = 40
output_dim = 12

Using device: mps


## Step 3: Dataset & DataLoaders

Same 12-class SPEECHCOMMANDS setup as the previous tutorials.

In [11]:
KEYWORDS      = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
LABEL_MAP     = {word: i for i, word in enumerate(KEYWORDS)}
SILENCE_LABEL = 10
UNKNOWN_LABEL = 11
TARGET_LEN    = 16000

class SPEECHCOMMANDS_12C(Dataset):
    """torchaudio SPEECHCOMMANDS mapped to 12 integer classes."""
    def __init__(self, root, url='speech_commands_v0.02',
                 folder_in_archive='SpeechCommands', download=False, subset=None):
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=root, url=url, folder_in_archive=folder_in_archive,
            download=download, subset=subset,
        )
        self.silence = []
        noise_dir = os.path.join(root, folder_in_archive, '_background_noise_')
        if os.path.exists(noise_dir):
            for fname in sorted(os.listdir(noise_dir)):
                if fname.endswith('.wav'):
                    wav, _ = torchaudio.load(os.path.join(noise_dir, fname))
                    for start in range(0, wav.shape[1] - TARGET_LEN, TARGET_LEN):
                        self.silence.append(wav[:, start:start + TARGET_LEN])

    @staticmethod
    def _fix_length(wav):
        n = wav.shape[1]
        if n < TARGET_LEN:
            return torch.nn.functional.pad(wav, (0, TARGET_LEN - n))
        return wav[:, :TARGET_LEN]

    def __len__(self):
        return len(self.base) + len(self.silence)

    def __getitem__(self, idx):
        if idx < len(self.base):
            waveform, sr, label, speaker_id, utt_num = self.base[idx]
            return self._fix_length(waveform), sr, LABEL_MAP.get(label, UNKNOWN_LABEL), speaker_id, utt_num
        return self.silence[idx - len(self.base)], 16000, SILENCE_LABEL, '', 0

def collate_fn(batch):
    waveforms = pad_sequence([b[0].squeeze(0) for b in batch], batch_first=True)
    return {'waveforms': waveforms, 'labels': torch.tensor([b[2] for b in batch])}

trainset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                               folder_in_archive='SpeechCommands',
                               download=download_option, subset='training')
validset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                               folder_in_archive='SpeechCommands',
                               download=download_option, subset='validation')
testset  = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                               folder_in_archive='SpeechCommands',
                               download=download_option, subset='testing')

class_weights  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4.6, 1/17]
sample_weights = [class_weights[label] for _, _, label, _, _ in trainset]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# num_workers=0 avoids pickling errors for classes defined in the notebook (__main__)
trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler,  collate_fn=collate_fn, num_workers=0)
validloader = DataLoader(validset, batch_size=batch_size, collate_fn=collate_fn, num_workers=0)
testloader  = DataLoader(testset,  batch_size=batch_size, collate_fn=collate_fn, num_workers=0)

print(f"Train: {len(trainset):,}  |  Val: {len(validset):,}  |  Test: {len(testset):,}")

Train: 84,843  |  Val: 9,981  |  Test: 11,005


## Step 4: Model

The nnAudio2 `MelSpectrogram` layer sits at the top of the model, exactly as in Parts 1–3. The BC-ResNet blocks below it expect a 4-D input `[B, 1, F, T]`, so we unsqueeze the channel dimension after log-compression.

To try a different architecture, replace the BC-ResNet blocks with your own `nn.Module` layers — everything else stays the same.

In [12]:
class SubSpectralNorm(nn.Module):
    def __init__(self, C, S, eps=1e-5):
        super().__init__()
        self.S, self.eps = S, eps
        self.bn = nn.BatchNorm2d(C * S)

    def forward(self, x):
        N, C, F, T = x.size()
        return self.bn(x.view(N, C * self.S, F // self.S, T)).view(N, C, F, T)


class BroadcastedBlock(nn.Module):
    def __init__(self, planes, dilation=1, stride=1, temp_pad=(0, 1)):
        super().__init__()
        self.freq_dw_conv  = nn.Conv2d(planes, planes, (3,1), padding=(1,0), groups=planes, dilation=dilation, stride=stride, bias=False)
        self.ssn1          = SubSpectralNorm(planes, 5)
        self.temp_dw_conv  = nn.Conv2d(planes, planes, (1,3), padding=temp_pad, groups=planes, dilation=dilation, stride=stride, bias=False)
        self.bn            = nn.BatchNorm2d(planes)
        self.channel_drop  = nn.Dropout2d(p=0.1)
        self.swish         = nn.SiLU()
        self.conv1x1       = nn.Conv2d(planes, planes, 1, bias=False)
        self.relu          = nn.ReLU(inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        identity = x
        out = self.ssn1(self.freq_dw_conv(x))
        aux = out
        out = self.bn(self.temp_dw_conv(out.mean(2, keepdim=True)))
        out = self.channel_drop(self.conv1x1(self.swish(out)))
        return self.relu(out + identity + aux)


class TransitionBlock(nn.Module):
    def __init__(self, inplanes, planes, dilation=1, stride=1, temp_pad=(0, 1)):
        super().__init__()
        self.conv1x1_1    = nn.Conv2d(inplanes, planes, 1, bias=False)
        self.bn1          = nn.BatchNorm2d(planes)
        self.freq_dw_conv = nn.Conv2d(planes, planes, (3,1), padding=(1,0), groups=planes, dilation=dilation, stride=stride, bias=False)
        self.ssn          = SubSpectralNorm(planes, 5)
        self.temp_dw_conv = nn.Conv2d(planes, planes, (1,3), padding=temp_pad, groups=planes, dilation=dilation, stride=stride, bias=False)
        self.bn2          = nn.BatchNorm2d(planes)
        self.channel_drop = nn.Dropout2d(p=0.5)
        self.swish        = nn.SiLU()
        self.conv1x1_2    = nn.Conv2d(planes, planes, 1, bias=False)
        self.relu         = nn.ReLU(inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        out = self.relu(self.bn1(self.conv1x1_1(x)))
        out = self.ssn(self.freq_dw_conv(out))
        aux = out
        out = self.channel_drop(self.conv1x1_2(self.swish(self.bn2(self.temp_dw_conv(out.mean(2, keepdim=True))))))
        return self.relu(aux + out)


class BCResNet(LightningModule):
    """BC-ResNet keyword spotter with nnAudio2 MelSpectrogram as the front-end."""

    def __init__(self, trainable_mel=False, trainable_STFT=False):
        super().__init__()
        self.mel = MelSpectrogram(
            sr=16000, n_fft=480, hop_length=160, n_mels=n_mels,
            fmin=0.0, norm=1,
            trainable_mel=trainable_mel, trainable_STFT=trainable_STFT,
            verbose=False,
        )
        self.conv1     = nn.Conv2d(1, 16, 5, stride=(2,1), padding=(2,2))
        self.block1_1  = TransitionBlock(16, 8)
        self.block1_2  = BroadcastedBlock(8)
        self.block2_1  = TransitionBlock(8,  12, stride=(2,1), dilation=(1,2), temp_pad=(0,2))
        self.block2_2  = BroadcastedBlock(12, dilation=(1,2), temp_pad=(0,2))
        self.block3_1  = TransitionBlock(12, 16, stride=(2,1), dilation=(1,4), temp_pad=(0,4))
        self.block3_2  = BroadcastedBlock(16, dilation=(1,4), temp_pad=(0,4))
        self.block3_3  = BroadcastedBlock(16, dilation=(1,4), temp_pad=(0,4))
        self.block3_4  = BroadcastedBlock(16, dilation=(1,4), temp_pad=(0,4))
        self.block4_1  = TransitionBlock(16, 20, dilation=(1,8), temp_pad=(0,8))
        self.block4_2  = BroadcastedBlock(20, dilation=(1,8), temp_pad=(0,8))
        self.block4_3  = BroadcastedBlock(20, dilation=(1,8), temp_pad=(0,8))
        self.block4_4  = BroadcastedBlock(20, dilation=(1,8), temp_pad=(0,8))
        self.conv2     = nn.Conv2d(20, 20, 5, groups=20, padding=(0,2))
        self.conv3     = nn.Conv2d(20, 32, 1, bias=False)
        self.conv4     = nn.Conv2d(32, output_dim, 1, bias=False)
        self.criterion = nn.CrossEntropyLoss()
        self._test_outputs = []

    def forward(self, x):
        spec = torch.log(self.mel(x) + 1e-10).unsqueeze(1)  # [B,1,F,T]
        out  = self.conv1(spec)
        out  = self.block1_2(self.block1_1(out))
        out  = self.block2_2(self.block2_1(out))
        out  = self.block3_4(self.block3_3(self.block3_2(self.block3_1(out))))
        out  = self.block4_4(self.block4_3(self.block4_2(self.block4_1(out))))
        out  = self.conv4(self.conv3(self.conv2(out)).mean(-1, keepdim=True))
        return out.squeeze(2).squeeze(2), spec.squeeze(1)   # [B,12], [B,F,T]

    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure=None):
        optimizer.step(closure=optimizer_closure)
        with torch.no_grad():
            torch.clamp_(self.mel.mel_basis, 0, 1)

    def _step(self, batch):
        logits, _ = self(batch['waveforms'])
        loss = self.criterion(logits, batch['labels'])
        acc  = (logits.argmax(-1) == batch['labels']).float().mean()
        return loss, acc

    def training_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'train_loss': loss, 'train_acc': acc}, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'val_loss': loss, 'val_acc': acc}, prog_bar=True)

    def test_step(self, batch, batch_idx):
        logits, _ = self(batch['waveforms'])
        loss = self.criterion(logits, batch['labels'])
        self.log('test_loss', loss, prog_bar=True)
        self._test_outputs.append({'logits': logits.detach().cpu(), 'labels': batch['labels'].detach().cpu()})

    def on_test_epoch_end(self):
        logits = torch.cat([o['logits'] for o in self._test_outputs])
        labels = torch.cat([o['labels'] for o in self._test_outputs])
        self._test_outputs.clear()
        preds = logits.argmax(-1)
        self.log('test_acc', (preds == labels).float().mean())
        for avg in ['micro', 'macro', 'weighted']:
            _, _, f1, _ = precision_recall_fscore_support(labels, preds, average=avg, zero_division=0)
            self.log(f'test_{avg}_f1', float(f1))

    def configure_optimizers(self):
        mel_params        = [p for n, p in self.named_parameters() if 'mel.' in n]
        classifier_params = [p for n, p in self.named_parameters() if 'mel.' not in n]
        return optim.Adam([
            {'params': classifier_params, 'lr': 1e-3},
            {'params': mel_params,        'lr': 1e-4},
        ])

model = BCResNet()
print(model)

BCResNet(
  (mel): MelSpectrogram(
    Mel filter banks size = (40, 241), trainable_mel=False
    (stft): STFT(n_fft=480, Fourier Kernel size=(241, 1, 480), iSTFT=False, trainable=False)
  )
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(2, 1), padding=(2, 2))
  (block1_1): TransitionBlock(
    (conv1x1_1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (freq_dw_conv): Conv2d(8, 8, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0), groups=8, bias=False)
    (ssn): SubSpectralNorm(
      (bn): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (temp_dw_conv): Conv2d(8, 8, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1), groups=8, bias=False)
    (bn2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (channel_drop): Dropout2d(p=0.5, inplace=False)
    (swish): SiLU()
    (conv1x1_2): Conv2d(8, 8, kern

## Step 5: Train & test

Lightning handles device placement. After training, `trainer.test()` reports accuracy and F1 scores on the held-out test set.

In [13]:
trainer = Trainer(
    accelerator=accelerator,
    devices=1,
    max_epochs=max_epochs,
    check_val_every_n_epoch=check_val_every_n_epoch,
    num_sanity_val_steps=num_sanity_val_steps,
)
trainer.fit(model, trainloader, validloader)
trainer.test(model, testloader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

   | Name      | Type             | Params | Mode 
--------------------------------------------------------
0  | mel       | Me

Sanity Checking DataLoader 0:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 1: 100%|██████████| 849/849 [00:31<00:00, 26.67it/s, v_num=9, train_loss=1.370, train_acc=0.525]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 849/849 [00:32<00:00, 26.53it/s, v_num=9, train_loss=0.391, train_acc=0.872, val_loss=0.733, val_acc=0.744]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 849/849 [00:32<00:00, 26.39it/s, v_num=9, train_loss=0.289, train_acc=0.905, val_loss=0.503, val_acc=0.829]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 7: 100%|██████████| 849/849 [00:31<00:00, 26.55it/s, v_num=9, train_loss=0.247, train_acc=0.918, val_loss=0.537, val_acc=0.816]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 9: 100%|██████████| 849/849 [00:32<00:00, 26.36it/s, v_num=9, train_loss=0.228, train_acc=0.925, val_loss=0.361, val_acc=0.879]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 11: 100%|██████████| 849/849 [00:32<00:00, 26.23it/s, v_num=9, train_loss=0.206, train_acc=0.931, val_loss=0.469, val_acc=0.8


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

After training you will see test scores including accuracy and micro/macro/weighted F1.

Trained weights are saved automatically to `lightning_logs/`. To visualise the learned basis functions, load your checkpoint in **Part 3** (Step 7) — the visualisation code works for any model that uses `self.mel`.

## Conclusion

Swapping in BC-ResNet required only replacing the classifier layers. The nnAudio2 `MelSpectrogram` front-end stayed unchanged — it is just an `nn.Module` that moves with the model and trains alongside it.

This is the end of the nnAudio2 tutorial series. We hope it helps you get started with GPU-accelerated, differentiable audio feature extraction in your own projects.